# S07 · Scoring a model honestly

In notebook 01 we drew a line but never asked *how good it is*. Here we do it properly. The trap is to grade the model on the very districts it learned from — flattering and a little dishonest, like testing a student on questions they had already seen. The honest way: learn on one part of the data, then judge on a part the model was **never allowed to see**.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GirishMKulkarni/applied-maths-in-industry-site/blob/main/sessions/S07/notebooks/02_first_house_price_model.ipynb)

*New here? Press each cell's play button, top to bottom, and read the plain-English note above it. Nothing to install on Colab.*

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on each cell, top to bottom, and read the plain-English note above it.
- Want the idea behind today in one page? Open `primers/least_squares_and_lines.md`.
- Already confident with code? Skip to the last **Your turn** cell.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook only uses numpy, pandas, matplotlib and scikit-learn,
# all of which Google Colab already ships, so there is nothing to install.
print("Setup complete - nothing to install.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

## Step 1 — the same real data as notebook 01

The data is real: **California housing**, one row per district, shipped inside `scikit-learn`. It downloads once, then is cached, so re-runs are instant. Each district records a few facts; the one we want to predict is `MedHouseVal`, the median house value (in units of **$100,000**).

In [ ]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True).frame
print("rows, columns:", data.shape)
data.head()

## Step 2 — the honest move: split into train and test

Here is the habit that matters more than any model. Before fitting, we hide **20%** of the districts in a test set and do not touch them. The model learns only from the other 80%. At the end we score it on the hidden 20% — districts it has never seen — the only fair measure of how it will do in real life.

`random_state=0` just makes the split the same for everyone.

In [ ]:
X = data[["MedInc"]]
y = data["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)

print("learn from :", X_train.shape[0], "districts")
print("judge on   :", X_test.shape[0], "hidden districts")

## Step 3 — fit on the training data only

Exactly the two lines from notebook 01 — but fitted on the training set alone. The test set stays in the drawer.

In [ ]:
model = LinearRegression().fit(X_train, y_train)

print("slope :", round(model.coef_[0], 3))
print("start :", round(model.intercept_, 3))

## Step 4 — score it on the hidden test data

Now open the drawer. We predict on the test districts and compare with what they were really worth, using two honest scores:

- **MAE** — the average miss, in the same units as price ($100,000s). Easiest to explain.
- **R²** — the share of the ups and downs the line explains. 1 is perfect, 0 is no better than always guessing the average, and it can even go negative.

In [ ]:
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("MAE:", round(mae, 3), " (off by about $" + format(round(mae * 100000), ",") + " on average)")
print("R2 :", round(r2, 3), " (share of the variation explained)")

## Step 5 — the picture that never lies: predicted vs actual

A single score can hide a lot. Always plot the model's prediction against the real value. Perfect predictions would sit exactly on the dashed diagonal; the spread around it is the model's error, in full view.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, predictions, s=6, alpha=0.2, color="#2E75B6")
lo, hi = y_test.min(), y_test.max()
plt.plot([lo, hi], [lo, hi], "--", color="#C0392B", label="perfect prediction")
plt.xlabel("real house value ($100,000s)")
plt.ylabel("predicted house value")
plt.title("Predicted vs actual on the hidden districts")
plt.legend()
plt.show()

### Stretch (optional) — give the model every column

One clue (income) only goes so far. Real models use every column at once — the same straight-line machine, just with a slope per feature. Watch R² improve when the model can see more.

In [ ]:
feature_cols = [c for c in data.columns if c != "MedHouseVal"]
Xa = data[feature_cols]
Xa_train, Xa_test, y_train, y_test = train_test_split(Xa, y, test_size=0.2, random_state=0)

full = LinearRegression().fit(Xa_train, y_train)
r2_full = r2_score(y_test, full.predict(Xa_test))

print("R2 with one feature  :", round(r2, 3))
print("R2 with all features :", round(r2_full, 3))

## Your turn (5-10 minutes)

1. Change `test_size` in Step 2 from `0.2` to `0.4` and re-run Steps 2-4. Do the scores move much?
2. In Step 2, swap the single feature `"MedInc"` for `"HouseAge"` and re-run. Is age a better or worse single predictor? Look at the R².
3. In a text cell, answer as if to a manager: *"on average, how many dollars is our one-feature model off by?"*

In [ ]:
# Your turn -- edit and re-run the cells above, or experiment here.


## What you just did

You trained a price model on real data and — the part that matters — judged it **honestly** on districts it had never seen, with MAE and R², and looked at the predicted-vs-actual picture. Scoring on held-out data is the single most important habit in machine learning; every later model in this course is judged the same way.

Next notebook: `03_learning_step_by_step.ipynb`, where we open the box and *watch* the model find the line — one downhill step at a time — and meet the single most important dial in modern machine learning: the **learning rate**.